### Setup

In [53]:
import pandas as pd
import os
from typhoon_ocr import ocr_document
from bs4 import BeautifulSoup
import re
from rapidfuzz import fuzz, process
from tqdm import tqdm

In [54]:
os.environ["TYPHOON_OCR_API_KEY"] = "sk-ogvTcIwhoNXX69zmxalfZwZXW8JU5YcohnT4ISIgLTTapvJQ"

In [55]:
SUBMISSION_PATH = "../data/submission_template.csv"

In [56]:
template_df = pd.read_csv(SUBMISSION_PATH)
template_df

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0
...,...,...,...,...,...
10048,party_list_34_11_53,party_list_34_11,53,ไทยพิทักษ์ธรรม,0
10049,party_list_34_11_54,party_list_34_11,54,ความหวังใหม่,0
10050,party_list_34_11_55,party_list_34_11,55,ไทยรวมไทย,0
10051,party_list_34_11_56,party_list_34_11,56,เพื่อบ้านเมือง,0


### EDA

In [57]:
template_df['party_name'].unique()

array(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
       'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
       'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
       'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
       'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
       'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
       'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
       'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
       'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', nan,
       'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
       'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
       'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
       'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
       'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
       'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธ

In [58]:
template_df.head()

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0


In [59]:
invalid_name = ["Unknown Party", "พรรคที่ 1 (ไม่ระบุชื่อ)", "ไม่ระบุ"]

NaN_df = template_df[
    template_df['party_name'].isna() | template_df['party_name'].isin(invalid_name)
]

NaN_df

,id,doc_id,row_num,party_name,votes
737,constituency_14_2_10,constituency_14_2,10,NaN,0
738,constituency_14_2_11,constituency_14_2,11,NaN,0
739,constituency_14_2_12,constituency_14_2,12,NaN,0
740,constituency_14_2_13,constituency_14_2,13,NaN,0
741,constituency_14_2_14,constituency_14_2,14,NaN,0
742,constituency_14_2_15,constituency_14_2,15,NaN,0
743,constituency_14_2_16,constituency_14_2,16,NaN,0
744,constituency_14_2_17,constituency_14_2,17,NaN,0
2698,party_list_10_3_1,party_list_10_3,1,NaN,0
2700,party_list_10_3_3,party_list_10_3,3,NaN,0


In [60]:
# 737 - 744 Correctly Null
template_df.iloc[737]

id            constituency_14_2_10
doc_id           constituency_14_2
row_num                         10
party_name                     NaN
votes                            0
Name: 737, dtype: object

In [61]:
# 2698
template_df.iloc[2698]

id            party_list_10_3_1
doc_id          party_list_10_3
row_num                       1
party_name                  NaN
votes                         0
Name: 2698, dtype: object

In [62]:
# 2700
template_df.iloc[2700]

id            party_list_10_3_3
doc_id          party_list_10_3
row_num                       3
party_name                  NaN
votes                         0
Name: 2700, dtype: object

In [63]:
# 6461
template_df.iloc[6461]

id                  party_list_21_4_1
doc_id                party_list_21_4
row_num                             1
party_name    พรรคที่ 1 (ไม่ระบุชื่อ)
votes                               0
Name: 6461, dtype: object

In [64]:
# 6892
template_df.iloc[6892]

id            party_list_24_2_33
doc_id           party_list_24_2
row_num                       33
party_name                   NaN
votes                          0
Name: 6892, dtype: object

In [65]:
# 7031
template_df.iloc[7031]

id            party_list_25_1_1
doc_id          party_list_25_1
row_num                       1
party_name                  NaN
votes                         0
Name: 7031, dtype: object

In [66]:
# 7033
template_df.iloc[7033]

id            party_list_25_1_3
doc_id          party_list_25_1
row_num                       3
party_name                  NaN
votes                         0
Name: 7033, dtype: object

In [67]:
# 8000
template_df.iloc[8000]

id            party_list_30_4_1
doc_id          party_list_30_4
row_num                       1
party_name        Unknown Party
votes                         0
Name: 8000, dtype: object

In [68]:
# 9426
template_df.iloc[9426]

id            party_list_33_2_1
doc_id          party_list_33_2
row_num                       1
party_name                  NaN
votes                         0
Name: 9426, dtype: object

In [69]:
fix_invalid = {
    2698: ('ไทยทรัพย์ทวี', 31),
    2700: ('ใหม่', 149),
    6461: ('ไทยทรัพย์ทวี', 473),
    6892: ('ประชาชาติ', 632),
    7031: ('ไทยทรัพย์ทวี', 2250),
    7033: ('ใหม่', 481),
    8000: ('ไทยทรัพย์ทวี', 420),
    9426: ('ไทยทรัพย์ทวี', 347)
}

In [70]:
# Fix invalid
for idx, (party, vote) in fix_invalid.items():
    template_df.loc[idx, 'party_name'] = party
    template_df.loc[idx, 'votes'] = vote

for idx, (party, vote) in fix_invalid.items():
    print(template_df.iloc[idx])
    print()

id            party_list_10_3_1
doc_id          party_list_10_3
row_num                       1
party_name         ไทยทรัพย์ทวี
votes                        31
Name: 2698, dtype: object

id            party_list_10_3_3
doc_id          party_list_10_3
row_num                       3
party_name                 ใหม่
votes                       149
Name: 2700, dtype: object

id            party_list_21_4_1
doc_id          party_list_21_4
row_num                       1
party_name         ไทยทรัพย์ทวี
votes                       473
Name: 6461, dtype: object

id            party_list_24_2_33
doc_id           party_list_24_2
row_num                       33
party_name             ประชาชาติ
votes                        632
Name: 6892, dtype: object

id            party_list_25_1_1
doc_id          party_list_25_1
row_num                       1
party_name         ไทยทรัพย์ทวี
votes                      2250
Name: 7031, dtype: object

id            party_list_25_1_3
doc_id          party_list_2

### Extraction

In [71]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [72]:
def extract_party_score_dict(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    table = soup.find("table")
    if table is None:
        raise ValueError("No <table> found")

    rows = table.find_all("tr")
    if not rows:
        return {}

    headers = [td.get_text(strip=True) for td in rows[0].find_all(["td", "th"])]

    # Candidate labels
    party_candidates = ["พรรคการเมือง", "ชื่อพรรคการเมือง", "สังกัดพรรคการเมือง"]
    score_candidates = ["ได้คะแนน"]

    party_idx = None
    score_idx = None
    best_party_score = 0
    best_score_score = 0

    for i, h in enumerate(headers):
        # compute best similarity across all candidates
        party_sim = max(fuzz.partial_ratio(h, c) for c in party_candidates)
        score_sim = max(fuzz.partial_ratio(h, c) for c in score_candidates)

        if party_sim > best_party_score:
            best_party_score = party_sim
            party_idx = i

        if score_sim > best_score_score:
            best_score_score = score_sim
            score_idx = i

    if best_party_score < 60 or best_score_score < 60:
        raise ValueError("Required columns not confidently found")

    result = {}

    for row in rows[1:]:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]

        if len(cols) <= max(party_idx, score_idx):
            continue

        key = cols[party_idx]
        value = cols[score_idx]

        try:
            value = thai_num_to_int(value)
        except Exception:
            continue  # safer: skip invalid rows

        result[key] = value

    return result

In [73]:
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        markdown = ocr_document(
            pdf_or_image_path=path
        )
        party_dict = extract_party_score_dict(markdown)
        return party_dict
    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [74]:
def merge_pages(*pages):
    merged = {}
    for page in pages:
        if page is None:
            continue
        for k, v in page.items():
            merged[k] = merged.get(k, 0) + v
    return merged

In [75]:
def assign_votes(df, vote_dict, threshold=80):
    # Remove non-party keys
    vote_dict = {
        k: v for k, v in vote_dict.items()
        if k != 'รวมคะแนนทั้งสิ้น'
    }

    keys = list(vote_dict.keys())

    def get_vote(party_name):
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.ratio
        )
        
        if match is None:
            return 0
        
        best_key, score, _ = match
        
        if score >= threshold:
            return vote_dict[best_key]
        return 0

    df['votes'] = df['party_name'].apply(get_vote)
    return df

In [76]:
submission_df = template_df.copy()

In [77]:
PREFIX = "../data/images/"

doc_ids = submission_df['doc_id'].unique()

In [78]:
for doc_id in tqdm(doc_ids, desc="Processing", unit="doc"):
    try:
        pages = []

        for i in range(1, 11):
            if i == 1:
                path = PREFIX + doc_id + ".png"
            else:
                path = PREFIX + doc_id + f"_page{i}.png"

            pages.append(extraction(path))

        vote_dict = merge_pages(*pages)

        mask = submission_df['doc_id'] == doc_id

        submission_df.loc[mask] = assign_votes(
            submission_df.loc[mask].copy(),
            vote_dict,
            threshold=80
        )

    except Exception as e:
        print(f"{doc_id}: {str(e)}")

Processing:   0%|          | 0/300 [00:00<?, ?doc/s]

No <table> found


Processing:   0%|          | 1/300 [00:19<1:37:08, 19.49s/doc]

No <table> found
No <table> found


Processing:   1%|          | 3/300 [01:21<2:29:11, 30.14s/doc]

...

Processing:  99%|█████████▉| 297/300 [4:55:37<03:27, 69.30s/doc]

No <table> found


Processing:  99%|█████████▉| 298/300 [4:57:11<02:33, 76.56s/doc]

No <table> found


Processing: 100%|██████████| 300/300 [4:59:24<00:00, 59.88s/doc]


In [79]:
submission_df

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,14813
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,14368
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,979
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,244
4,constituency_10_1_5,constituency_10_1,5,พลวัต,351
...,...,...,...,...,...
10048,party_list_34_11_53,party_list_34_11,53,ไทยพิทักษ์ธรรม,14
10049,party_list_34_11_54,party_list_34_11,54,ความหวังใหม่,41
10050,party_list_34_11_55,party_list_34_11,55,ไทยรวมไทย,0
10051,party_list_34_11_56,party_list_34_11,56,เพื่อบ้านเมือง,29


In [80]:
submission_df.to_csv("submission.csv")